In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pickle
import os
import json
import scipy.interpolate as interp
import pathlib

In [3]:

dir = r"/Volumes/ESSD/BatteryLife" if os.name == 'posix' else r"D:\BatteryLife"
content = os.listdir(dir)
print(f"###### \n ---- Raw Content ---- \n{content} \n######")
folders = [f for f in content if os.path.isdir(os.path.join(dir, f)) and (f != "Life labels" and f != "READMEs")]
print(f"###### \n ---- Battery Folders ---- \n{folders} \n######")

###### 
 ---- Raw Content ---- 
['CALB', 'CALCE', 'HNEI', 'HUST', 'ISU_ILCC', 'Life labels', 'MATR', 'MICH', 'MICH_EXP', 'NA-ion', 'README.md', 'READMEs', 'RWTH', 'SDU', 'SNL', 'Stanford', 'Stanford_2', 'Tongji', 'UL_PUR', 'XJTU', 'ZN-coin', '.DS_Store', '._.DS_Store', '._README.md'] 
######
###### 
 ---- Battery Folders ---- 
['CALB', 'CALCE', 'HNEI', 'HUST', 'ISU_ILCC', 'MATR', 'MICH', 'MICH_EXP', 'NA-ion', 'RWTH', 'SDU', 'SNL', 'Stanford', 'Stanford_2', 'Tongji', 'UL_PUR', 'XJTU', 'ZN-coin'] 
######


In [4]:

def get_length(dir):
    content = os.listdir(dir)
    length = 0
    for f in content:
        if f.startswith("._") or f.startswith(".DS_Store"):
            continue
        with open(os.path.join(dir, f), 'r') as file:
            data = json.load(file)
            length += len(data)
    return length


        

In [5]:
content = os.listdir(dir)
print(f"###### \n ---- Content ---- \n{content} \n######")
folders = [f for f in content if os.path.isdir(os.path.join(dir, f)) and (f != "Life labels" and f != "READMEs") and not f.startswith("._") and not f.startswith(".DS_Store")]
pkl_files = [[] for _ in folders]
for i, folder in enumerate(folders) :
    folder_path = os.path.join(dir, folder)
    for file in os.listdir(folder_path):
        if file.endswith('.pkl'):
            pkl_files[i].append(file)
print(len(pkl_files))

###### 
 ---- Content ---- 
['CALB', 'CALCE', 'HNEI', 'HUST', 'ISU_ILCC', 'Life labels', 'MATR', 'MICH', 'MICH_EXP', 'NA-ion', 'README.md', 'READMEs', 'RWTH', 'SDU', 'SNL', 'Stanford', 'Stanford_2', 'Tongji', 'UL_PUR', 'XJTU', 'ZN-coin', '.DS_Store', '._.DS_Store', '._README.md'] 
######
18


In [ ]:
labels = os.path.join(dir, "Life labels")
def get_dict(labels):
    dict_labels = {}
    content = os.listdir(labels)
    length = 0
  
    for f in content:
        if f.startswith("._") or f.startswith(".DS_Store"):
          continue
        with open(os.path.join(labels, f), 'r') as file:
            data = json.load(file)
            if 'Tongji' in f:
               temp_dict = {}
               for k, v in data.items():
                    k_new = k.replace("#", "-")
                    temp_dict.update({k_new:v})
               data = temp_dict   
            dict_labels.update(data)
    k= list(dict_labels.keys())
    v= list(dict_labels.values())
    return k,v


In [15]:
k,v = get_dict(labels)
print(k)

['CALB_0_B182.pkl', 'CALB_0_B183.pkl', 'CALB_0_B184.pkl', 'CALB_0_B185.pkl', 'CALB_0_B187.pkl', 'CALB_0_B188.pkl', 'CALB_0_B189.pkl', 'CALB_0_B190.pkl', 'CALB_25_T25-1.pkl', 'CALB_25_T25-2.pkl', 'CALB_35_B173.pkl', 'CALB_35_B174.pkl', 'CALB_35_B175.pkl', 'CALB_35_B222.pkl', 'CALB_35_B223.pkl', 'CALB_35_B224.pkl', 'CALB_35_B227.pkl', 'CALB_35_B228.pkl', 'CALB_35_B229.pkl', 'CALB_35_B230.pkl', 'CALB_35_B247.pkl', 'CALB_35_B248.pkl', 'CALB_35_B249.pkl', 'CALB_35_B250.pkl', 'CALB_45_B253.pkl', 'CALB_45_B255.pkl', 'CALB_45_B256.pkl', 'CALCE_CS2_38.pkl', 'CALCE_CX2_36.pkl', 'CALCE_CS2_37.pkl', 'CALCE_CS2_36.pkl', 'CALCE_CS2_34.pkl', 'CALCE_CS2_33.pkl', 'CALCE_CX2_38.pkl', 'CALCE_CS2_35.pkl', 'CALCE_CX2_35.pkl', 'CALCE_CX2_34.pkl', 'CALCE_CX2_33.pkl', 'CALCE_CX2_37.pkl', 'CALCE_CX2_16.pkl', 'HNEI_18650_NMC_LCO_25C_0-100_0.5-1.5C_o.pkl', 'HNEI_18650_NMC_LCO_25C_0-100_0.5-1.5C_l.pkl', 'HNEI_18650_NMC_LCO_25C_0-100_0.5-1.5C_c.pkl', 'HNEI_18650_NMC_LCO_25C_0-100_0.5-1.5C_n.pkl', 'HNEI_18650_NMC_L

In [7]:
def get_cycle_data(data):
    arr_total = np.zeros((50, 3, 300))
    for i in range(50):
      scaled_time = np.array(data['cycle_data'][i]['time_in_s']) - data['cycle_data'][i]['time_in_s'][0]
      t_new = np.linspace(0, scaled_time[-1], num=300)  
      arr = interp.interp1d(scaled_time, data['cycle_data'][i]['current_in_A'], kind='linear')
      arr_2 = interp.interp1d(scaled_time, data['cycle_data'][i]['voltage_in_V'], kind='linear')
      arr_3 =  interp.interp1d(scaled_time, data['cycle_data'][i]['charge_capacity_in_Ah'], kind='linear')
      arr_4 =  interp.interp1d(scaled_time, data['cycle_data'][i]['discharge_capacity_in_Ah'], kind='linear')
      current_interpolated = arr(t_new)/data['nominal_capacity_in_Ah']
      voltage_interpolated = arr_2(t_new)/data['max_voltage_limit_in_V']
      charge_capacity_interpolated = arr_3(t_new)/data['nominal_capacity_in_Ah']
      discharge_capacity_interpolated = arr_4(t_new)/data['nominal_capacity_in_Ah']
      x = np.searchsorted(discharge_capacity_interpolated, 0, side='right')
      capacity_interpolated = np.concatenate((charge_capacity_interpolated[:x], charge_capacity_interpolated[-1] - discharge_capacity_interpolated[x:]))
      arr_total[i] = np.stack((current_interpolated, voltage_interpolated, capacity_interpolated), axis=0)
    return arr_total

In [8]:
def get_location(file, root_dir):
    for loc in pathlib.Path(root_dir).rglob(file):
        return loc

In [9]:
class BatteryLifeDataset(Dataset):
    def __init__(self, root_dir, transform=None, target_transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.target_transform = target_transform
        self.cycles_file, self.soh = get_dict(os.path.join(root_dir, "Life labels"))
        
    def __len__(self):
        return len(self.soh)
    
    def __getitem__(self, idx):
     file_name = self.cycles_file[idx]
     path = get_location(file_name, self.root_dir)

     if path is None:
        raise ValueError(f"Missing file: {file_name}")

     with open(path, 'rb') as file:
        data = pickle.load(file)

     features = get_cycle_data(data)
     label = self.soh[idx]

     return (
        torch.tensor(features, dtype=torch.float32),
        torch.tensor(label, dtype=torch.float32)
     )


In [10]:
instance = BatteryLifeDataset(root_dir=dir)
print(f"Length of dataset: {len(instance)}")
for i in range(500):
    ar, label = instance[i]
    print(f"Feature shape: {ar.shape}")
    print(f"Label: {label}")

Length of dataset: 1208
Feature shape: torch.Size([50, 3, 300])
Label: 107.0
Feature shape: torch.Size([50, 3, 300])
Label: 105.0
Feature shape: torch.Size([50, 3, 300])
Label: 176.0
Feature shape: torch.Size([50, 3, 300])
Label: 161.0
Feature shape: torch.Size([50, 3, 300])
Label: 189.0
Feature shape: torch.Size([50, 3, 300])
Label: 123.0
Feature shape: torch.Size([50, 3, 300])
Label: 164.0
Feature shape: torch.Size([50, 3, 300])
Label: 104.0
Feature shape: torch.Size([50, 3, 300])
Label: 792.0
Feature shape: torch.Size([50, 3, 300])
Label: 980.0
Feature shape: torch.Size([50, 3, 300])
Label: 1233.0
Feature shape: torch.Size([50, 3, 300])
Label: 1272.0
Feature shape: torch.Size([50, 3, 300])
Label: 1352.0
Feature shape: torch.Size([50, 3, 300])
Label: 1260.0
Feature shape: torch.Size([50, 3, 300])
Label: 1267.0
Feature shape: torch.Size([50, 3, 300])
Label: 1283.0
Feature shape: torch.Size([50, 3, 300])
Label: 1241.0
Feature shape: torch.Size([50, 3, 300])
Label: 1243.0
Feature shape:

KeyboardInterrupt: 

In [11]:
train_dataset = DataLoader(BatteryLifeDataset(root_dir=dir), batch_size=32, shuffle=True)

In [12]:
train_features, train_labels = next(iter(train_dataset))
train_features, train_labels = next(iter(train_dataset))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels}")

ValueError: Missing file: Tongji2_CY25-05_1-#5.pkl

In [27]:
x= torch.linspace(0, 10, steps=5)
y = torch.sin(x)
print(y)
p = torch.tensor([1, 2, 3])
t = x.unsqueeze(-1).pow(p)
t

tensor([ 0.0000,  0.5985, -0.9589,  0.9380, -0.5440])


tensor([[   0.0000,    0.0000,    0.0000],
        [   2.5000,    6.2500,   15.6250],
        [   5.0000,   25.0000,  125.0000],
        [   7.5000,   56.2500,  421.8750],
        [  10.0000,  100.0000, 1000.0000]])

In [44]:
tensor_1 = torch.tensor([[[1,2,4,5],[3,4,5,5],[6,5,3,4]],[[1,2,4,3],[3,4,5,5],[6,5,3,3]]])
tensor_1.shape
tensor_2 = tensor_1.reshape(2,-1)
print(tensor_2)


tensor([[1, 2, 4, 5, 3, 4, 5, 5, 6, 5, 3, 4],
        [1, 2, 4, 3, 3, 4, 5, 5, 6, 5, 3, 3]])
